# Corner-WFS Donut Selection Stages & Interactive Wavefront Inspector

**Author:** Aaron Roodman
**Date Created:** 2026-06-13
**Last Modified:** 2026-06-13
**Status:** In Progress
**Keywords:** WFS, corner wavefront sensor, donut selection, ts_wep, zernikes, wavefront, interactive

## Description

For each corner-WFS half-chip (intra / extra), show the **post-ISR image** with
**circles overlaid** on the detected donuts, coloured by ts_wep pipeline stage
(selected → fit → used). Then provide an **interactive tool**: click on a donut to
read out its **fitted wavefront** (Zernike coefficients + reconstructed OPD map).

Stage definitions (join key `donut_id`):
- **selected** — in `donutTable` (round-1 direct detection).
- **fit** — member of a pair in `aggregateAOSVisitTableRaw`.
- **used** — member of a pair with `used == True`.

See `ts_wep_cwfs_dataflow.md` for the full product flow.

**Output:** a per-visit PDF of image+circle overlays in `wfs/output/`, an extended
`aggregateDonutTable` (ECSV), and an interactive in-notebook inspector.

**Based on:** AOS `cwfs` reductions in
`LSSTCam/runs/aos/cwfs/danish_1_0/wep_17_3_0/dv_4_2_0/bin_x2` (repo `/repo/main`).

## Change Log

| Date | Author | Description |
|------|--------|-------------|
| 2026-06-13 | Aaron Roodman | Initial donut selection-stages notebook (per-CCD selected/fit/used + aggregate table) |
| 2026-06-13 | Aaron Roodman | Overlay stage circles on the CWFS images (open circles, colour=stage, radius>donut); add interactive click-to-inspect fitted-wavefront tool |

## Table of Contents

1. [Parameters](#params)
2. [Setup & Imports](#setup)
3. [Helper Functions](#functions)
4. [Data Access](#data)
5. [Select Visits](#select)
6. [Image + Stage-Circle Overlays](#plots)
7. [Aggregate Table (fit donuts + used flag)](#table)
8. [Interactive Wavefront Inspector](#interactive)

<a id='params'></a>
## Parameters

In [ ]:
# ============================================================
# Parameters — All configurable values collected here
# ============================================================

butler_repo = "/repo/main"
collection = "LSSTCam/runs/aos/cwfs/danish_1_0/wep_17_3_0/dv_4_2_0/bin_x2"
dataset_type = "post_isr_image"
instrument = "LSSTCam"

day_obs = 20260327               # night to inspect
max_visits = 3                   # how many visits to render (None = all on the night)
start_index = 0

# Corner wavefront-sensor half-chips. SW0 = extra-focal, SW1 = intra-focal;
# the paired fit products (zernikes / aggregateAOSVisitTableRaw) live on SW0.
WFS_DETECTORS = {
    191: "R00_SW0", 192: "R00_SW1",
    195: "R04_SW0", 196: "R04_SW1",
    199: "R40_SW0", 200: "R40_SW1",
    203: "R44_SW0", 204: "R44_SW1",
}
DEFOCAL = {"SW0": "extra", "SW1": "intra"}

# ---- Circle overlay ----------------------------------------------------
# Typical donuts are ~150 px across; draw circles a bit larger so the donut
# sits comfortably inside. Open circles (no fill); stage shown by colour only.
donut_diam_px = 150
circle_radius_px = 90
circle_lw = 1.5
color_selected = "deepskyblue"   # detected, not fit
color_fit = "orange"             # fit but not used
color_used = "lime"              # used in the averaged estimate

# ---- Image stretch (sky-tuned) -----------------------------------------
sky_lo_sigma = 2.0
sky_hi_sigma = 8.0
asinh_a = 0.1
cmap = "gray"

# ---- Interactive inspector ---------------------------------------------
inspect_visit = None             # None -> first rendered visit
inspect_detector = 191           # which half-chip to load in the inspector
select_tol_px = 120              # click must be within this many px of a centroid
pupil_obsc = 0.612               # LSST pupil obscuration (inner/outer radius)
wf_npix = 256                    # OPD map sampling

# ---- Layout / output ---------------------------------------------------
output_dir = "output"
output_pdf = None                # None -> output/wfs_donut_stages_{day_obs}.pdf
save_table = True
panel_width = 5.0
title_fontsize = 9
dpi = 150


<a id='setup'></a>
## Setup & Imports

In [ ]:
import os
import sys
from pathlib import Path

import numpy as np
import matplotlib.pyplot as plt
from matplotlib.patches import Circle
from matplotlib.lines import Line2D
from matplotlib.backends.backend_pdf import PdfPages
from astropy.stats import sigma_clipped_stats
from astropy.table import vstack
from astropy.visualization import AsinhStretch, ImageNormalize
import galsim

import lsst.daf.butler as dafButler

sys.path.insert(0, str(Path.cwd().parent))
from common.utils import setup_plotting

setup_plotting()
os.makedirs(output_dir, exist_ok=True)

# Noll indices carried in the zk_* arrays (length 21): Z4-Z19, then Z22-Z26.
NOLL_INDICES = [4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19,
                22, 23, 24, 25, 26]


<a id='functions'></a>
## Helper Functions

In [ ]:
def list_visits(butler, collection, instrument, day_obs):
    """Visits on a night that were fit (have an aggregateAOSVisitTableRaw)."""
    refs = butler.query_datasets(
        "aggregateAOSVisitTableRaw", collections=collection,
        where=f"instrument='{instrument}' and exposure.day_obs={day_obs}",
        explain=False, limit=None)
    return sorted({r.dataId["visit"] for r in refs})


def load_aos_visit_table(butler, collection, instrument, visit):
    """Per-visit fit-pair table (one row per fit pair, with `used`)."""
    return butler.get("aggregateAOSVisitTableRaw", collections=collection,
                      dataId={"instrument": instrument, "visit": visit})


def stage_sets(aos_table):
    """(fit_ids, used_ids) sets of donut_id from an aggregateAOSVisitTableRaw."""
    intra = np.asarray(aos_table["donut_id_intra"])
    extra = np.asarray(aos_table["donut_id_extra"])
    used = np.asarray(aos_table["used"]).astype(bool)
    return (set(intra) | set(extra)), (set(intra[used]) | set(extra[used]))


def load_image(butler, collection, dataset_type, instrument, visit, detector):
    """Post-ISR image array (electrons) for one half-chip.

    `post_isr_image` is keyed by `exposure` (the donut tables use `visit`); for
    these single-snap CWFS visits the two ids are equal.
    """
    exp = butler.get(dataset_type, collections=collection,
                     dataId={"instrument": instrument, "exposure": visit,
                             "detector": detector})
    return np.asarray(exp.image.array, dtype=float)


def load_donut_table(butler, collection, instrument, visit, detector):
    """Round-1 selected donuts for one detector, or None."""
    try:
        return butler.get("donutTable", collections=collection,
                          dataId={"instrument": instrument, "visit": visit,
                                  "detector": detector})
    except Exception:
        return None


def sky_norm(data, lo_sigma=2.0, hi_sigma=8.0, asinh_a=0.1):
    """ImageNormalize (asinh) centred on the sigma-clipped sky level."""
    finite = data[np.isfinite(data)]
    _, med, std = sigma_clipped_stats(finite, sigma=3.0, maxiters=5)
    return ImageNormalize(vmin=med - lo_sigma * std, vmax=med + hi_sigma * std,
                          stretch=AsinhStretch(a=asinh_a))


def stage_of(donut_id, fit_ids, used_ids):
    """'used' | 'fit' | 'selected' for one donut_id."""
    if donut_id in used_ids:
        return "used"
    if donut_id in fit_ids:
        return "fit"
    return "selected"


def overlay_circles(ax, donut_table, fit_ids, used_ids, *, radius, lw,
                    colors, counts=None):
    """Draw an open circle on each donut, coloured by stage."""
    if donut_table is None:
        return
    ids = np.asarray(donut_table["donut_id"])
    x = np.asarray(donut_table["centroid_x"]); y = np.asarray(donut_table["centroid_y"])
    for i, did in enumerate(ids):
        st = stage_of(did, fit_ids, used_ids)
        if counts is not None:
            counts[st] += 1
        ax.add_patch(Circle((x[i], y[i]), radius, fill=False, ec=colors[st],
                            lw=lw, alpha=0.9))


def pair_row_for(aos_table, donut_id):
    """Return the aggregateAOSVisitTableRaw row whose pair contains donut_id, else None."""
    intra = np.asarray(aos_table["donut_id_intra"])
    extra = np.asarray(aos_table["donut_id_extra"])
    m = (intra == donut_id) | (extra == donut_id)
    idx = np.where(m)[0]
    return aos_table[int(idx[0])] if len(idx) else None


def zernike_opd(zk_microns, obsc=0.612, npix=256):
    """Reconstruct the wavefront OPD map (microns) from the zk_* Noll coefficients."""
    coef = np.zeros(max(NOLL_INDICES) + 1)
    for k, j in enumerate(NOLL_INDICES):
        coef[j] = zk_microns[k]
    Z = galsim.zernike.Zernike(coef, R_outer=1.0, R_inner=obsc)
    g = np.linspace(-1.0, 1.0, npix)
    xx, yy = np.meshgrid(g, g)
    opd = Z.evalCartesian(xx, yy)
    r = np.hypot(xx, yy)
    opd[(r > 1.0) | (r < obsc)] = np.nan
    return opd


def zernike_text(zk_microns, frame="CCS"):
    """Formatted Noll-j / value (nm) table for a zk_* coefficient array."""
    lines = [f"  Noll  Z(nm)  [{frame}]"]
    for k, j in enumerate(NOLL_INDICES):
        lines.append(f"  Z{j:<3d} {1000.0 * zk_microns[k]:+8.1f}")
    return "\n".join(lines)


<a id='data'></a>
## Data Access

In [ ]:
butler = dafButler.Butler(butler_repo, collections=collection)
print("Repo:", butler_repo, "| Collection:", collection)


<a id='select'></a>
## Select Visits

In [ ]:
all_visits = list_visits(butler, collection, instrument, day_obs)
print(f"day_obs {day_obs}: {len(all_visits)} fit visits")
visits = all_visits[start_index:]
if max_visits is not None:
    visits = visits[:max_visits]
print("Rendering visits:", visits)


<a id='plots'></a>
## Image + Stage-Circle Overlays

In [ ]:
rafts = sorted({n.split("_")[0] for n in WFS_DETECTORS.values()})
name_to_det = {v: k for k, v in WFS_DETECTORS.items()}
STAGE_COLORS = {"selected": color_selected, "fit": color_fit, "used": color_used}

legend_handles = [
    Line2D([0], [0], marker="o", mfc="none", mec=color_selected, ms=10, mew=1.6,
           ls="none", label="selected (round 1)"),
    Line2D([0], [0], marker="o", mfc="none", mec=color_fit, ms=10, mew=1.6,
           ls="none", label="fit (in a pair)"),
    Line2D([0], [0], marker="o", mfc="none", mec=color_used, ms=10, mew=1.6,
           ls="none", label="used (used pair)"),
]


def plot_visit_overlay(visit, aos_table):
    """4x2 grid of CWFS images with stage circles overlaid."""
    fit_ids, used_ids = stage_sets(aos_table)
    panel_h = panel_width * 2000.0 / 4072.0
    fig, axes = plt.subplots(len(rafts), 2,
                             figsize=(2 * panel_width, len(rafts) * panel_h),
                             constrained_layout=True)
    axes = np.atleast_2d(axes)
    fig.set_constrained_layout_pads(w_pad=0.01, h_pad=0.01, wspace=0.01, hspace=0.05)
    for i, raft in enumerate(rafts):
        for j, sw in enumerate(("SW0", "SW1")):
            ax = axes[i, j]; name = f"{raft}_{sw}"; det = name_to_det[name]
            img = load_image(butler, collection, dataset_type, instrument, visit, det)
            ax.imshow(img, origin="lower", cmap=cmap, norm=sky_norm(
                img, sky_lo_sigma, sky_hi_sigma, asinh_a),
                aspect="equal", interpolation="nearest")
            dt = load_donut_table(butler, collection, instrument, visit, det)
            counts = dict(selected=0, fit=0, used=0)
            overlay_circles(ax, dt, fit_ids, used_ids, radius=circle_radius_px,
                            lw=circle_lw, colors=STAGE_COLORS, counts=counts)
            nsel = counts["selected"] + counts["fit"] + counts["used"]
            ax.set_title(f"{name} ({det}, {DEFOCAL[sw]})  sel={nsel} "
                         f"fit={counts['fit']+counts['used']} used={counts['used']}",
                         fontsize=title_fontsize)
            ax.set_xticks([]); ax.set_yticks([])
    fig.legend(handles=legend_handles, loc="upper right", fontsize=8, ncol=3)
    fig.suptitle(f"Visit {visit} — corner-WFS donut selection stages",
                 fontsize=title_fontsize + 3)
    return fig


pdf_path = Path(output_pdf) if output_pdf else \
    Path(output_dir) / f"wfs_donut_stages_{day_obs}.pdf"

agg_tables = []
with PdfPages(pdf_path) as pdf:
    for visit in visits:
        aos = load_aos_visit_table(butler, collection, instrument, visit)
        fig = plot_visit_overlay(visit, aos)
        pdf.savefig(fig, dpi=dpi)
        plt.show()
        # extended aggregateDonutTable for the table section
        fit_ids, used_ids = stage_sets(aos)
        agg = butler.get("aggregateDonutTable", collections=collection,
                         dataId={"instrument": instrument, "visit": visit})
        ids = np.asarray(agg["donut_id"])
        agg["visit"] = visit
        agg["fit"] = np.array([d in fit_ids for d in ids])
        agg["used"] = np.array([d in used_ids for d in ids])
        agg_tables.append(agg)
        print(f"visit {visit}: fit donuts={len(fit_ids)}, used donuts={len(used_ids)}")

print(f"\nWrote {len(visits)} page(s) to {pdf_path}")


<a id='table'></a>
## Aggregate Table (fit donuts + used flag)

In [ ]:
aggregate = vstack(agg_tables, metadata_conflicts="silent") if agg_tables else None
if aggregate is not None:
    print(f"aggregate rows: {len(aggregate)}  "
          f"(fit={int(np.sum(aggregate['fit']))}, used={int(np.sum(aggregate['used']))})")
    fit_donuts = aggregate[aggregate["fit"]]
    cols = [c for c in ["visit", "detector", "donut_id", "centroid_x", "centroid_y",
                        "thx_OCS", "thy_OCS", "snr", "fit", "used"]
            if c in aggregate.colnames]
    print(fit_donuts[cols][:12])
    if save_table:
        out = Path(output_dir) / f"wfs_donut_aggregate_{day_obs}.ecsv"
        aggregate.write(out, overwrite=True)
        print("\nWrote table:", out)


<a id='interactive'></a>
## Interactive Wavefront Inspector

Click on a donut in the image to read out its **fitted wavefront**. Requires the
interactive matplotlib backend (`ipympl`); run the cell below first. Clicking a
**fit/used** donut shows the pair's Zernike coefficients (text) and reconstructed OPD
map; clicking a **selected-only** donut (not fit) reports that no fit exists.

In [ ]:
%matplotlib widget

In [ ]:
class DonutInspector:
    """Click a donut on one half-chip to display its fitted wavefront."""

    def __init__(self, visit, detector):
        self.visit = visit
        self.det = detector
        self.name = WFS_DETECTORS[detector]
        self.img = load_image(butler, collection, dataset_type, instrument,
                              visit, detector)
        self.dt = load_donut_table(butler, collection, instrument, visit, detector)
        self.ids = np.asarray(self.dt["donut_id"])
        self.x = np.asarray(self.dt["centroid_x"])
        self.y = np.asarray(self.dt["centroid_y"])
        self.aos = load_aos_visit_table(butler, collection, instrument, visit)
        self.fit_ids, self.used_ids = stage_sets(self.aos)

        self.fig, (self.ax_img, self.ax_wf) = plt.subplots(
            1, 2, figsize=(12, 5), constrained_layout=True)
        self.ax_img.imshow(self.img, origin="lower", cmap=cmap,
                           norm=sky_norm(self.img, sky_lo_sigma, sky_hi_sigma, asinh_a),
                           aspect="equal", interpolation="nearest")
        overlay_circles(self.ax_img, self.dt, self.fit_ids, self.used_ids,
                        radius=circle_radius_px, lw=circle_lw, colors=STAGE_COLORS)
        self.ax_img.set_title(f"{self.name} ({detector}) — click a donut", fontsize=10)
        self.ax_img.set_xticks([]); self.ax_img.set_yticks([])
        self.ax_wf.set_title("fitted wavefront", fontsize=10)
        self.ax_wf.axis("off")
        self.marker, = self.ax_img.plot([], [], "x", color="red", ms=12, mew=2)
        self.cid = self.fig.canvas.mpl_connect("button_press_event", self.on_click)

    def on_click(self, event):
        if event.inaxes is not self.ax_img or event.xdata is None:
            return
        d = np.hypot(self.x - event.xdata, self.y - event.ydata)
        i = int(np.argmin(d))
        if d[i] > select_tol_px:
            print(f"no donut within {select_tol_px} px of "
                  f"({event.xdata:.0f}, {event.ydata:.0f})")
            return
        did = self.ids[i]
        self.marker.set_data([self.x[i]], [self.y[i]])
        st = stage_of(did, self.fit_ids, self.used_ids)
        row = pair_row_for(self.aos, did)
        self.ax_wf.clear()
        if row is None:
            self.ax_wf.axis("off")
            self.ax_wf.set_title(f"{did}\nstage={st} — not fit (no wavefront)",
                                 fontsize=10)
            print(f"{did}: stage={st} (not fit)")
        else:
            zk = np.asarray(row["zk_CCS"])
            opd = zernike_opd(zk, obsc=pupil_obsc, npix=wf_npix)
            im = self.ax_wf.imshow(opd, origin="lower", cmap="RdBu_r",
                                   extent=[-1, 1, -1, 1])
            self.ax_wf.set_aspect("equal"); self.ax_wf.axis("off")
            self.ax_wf.set_title(
                f"{did}\nstage={st}, used={bool(row['used'])}  "
                f"PV={np.nanmax(opd)-np.nanmin(opd):.3f} µm", fontsize=10)
            if not hasattr(self, "_cb"):
                self._cb = self.fig.colorbar(im, ax=self.ax_wf, fraction=0.046,
                                             label="OPD [µm]")
            print(f"\n=== {did}  (stage={st}, used={bool(row['used'])}) ===")
            print(f"pair: intra={row['donut_id_intra']}  extra={row['donut_id_extra']}")
            print(f"snr intra/extra: {row['snr_intra']:.0f} / {row['snr_extra']:.0f}")
            print(zernike_text(zk, frame="CCS"))
        self.fig.canvas.draw_idle()


inspector = DonutInspector(inspect_visit or visits[0], inspect_detector)
